### Purpose
This notebook validates the processed dataset before feature engineering and modeling.
<br> --- --- --- <br>
Este notebook valida o dataset processado antes da etapa de feature engineering e modelagem.
<br>
📝 Language: Technical documentation is maintained in English to ensure consistency and ease of maintenance. Bilingual support (Portuguese/English) is available exclusively within the notebooks.
<br> --- --- --- <br>
📝 Idioma: A documentação é mantida em inglês para garantir consistência técnica e evitar retrabalho na documentação. Isto aplica-se somente aos notebooks.

##### ⚙️The project follows this logic:
**raw data -> data quality -> feature engineering -> anomaly detection -> baseline model -> hybrid model -> performance benchmarking**

##### ⚙️Execution order
📝01_data_loading.ipynb -> 📝02_data_quality.ipynb -> 📝03_feature_engineering.ipynb -> 📝04_anomaly_detection.ipynb -> 
📝05_BaseLine.ipynb -> 📝06_HybridModel.ipynb -> 📝07_performance_benchmarking.ipynb 


### Main Components
- `ParquetRepository`
- `DataQuality`
- `DATA_PROCESSED`

### What the Notebook Does
- Loads 📦`beverage_sales_processed.parquet`
- Instantiates the `DataQuality` class
- Runs a full quality analysis using `run_full_analysis(tolerance=0.1)`
- Prints the validation report
- Displays a sample of total price inconsistencies
- Generates an executive summary
- Prints DataFrame column types
<br> --- --- --- <br>
- Carrega 📦`beverage_sales_processed.parquet`
- Instancia a classe `DataQuality`
- Executa uma análise completa com `run_full_analysis(tolerance=0.1)`
- Exibe o relatório de validação
- Mostra uma amostra de inconsistências de preço total
- Gera um resumo executivo
- Exibe os tipos das colunas do DataFrame

### Input
- 📦`beverage_sales_processed.parquet`

### Output
- Quality validation report in notebook output
- Executive summary in notebook output
- Sample of inconsistent records in notebook output
<br> --- --- --- <br>
- Relatório de qualidade exibido no notebook
- Resumo executivo exibido no notebook
- Amostra de registros inconsistentes exibida no notebook

### Typical Checks Expected in This Stage
Depending on the implementation of `DataQuality`, this stage may validate:
- Missing values
- Negative values
- Discount rules
- Duplicated records
- Invalid dates
- Inconsistencies between total price and expected calculated price
<br> --- --- --- <br>
Dependendo da implementação da classe `DataQuality`, esta etapa pode validar:
- Valores ausentes
- Valores negativos
- Regras de desconto
- Registros duplicados
- Datas inválidas
- Inconsistências entre preço total e preço total esperado

### Why This Step Matters
- Prevents low-quality records from affecting feature engineering
- Supports trust in the modeling pipeline
- Makes business rules more explicit
<br> --- --- --- <br>
- Evita que registros de baixa qualidade impactem o feature engineering
- Dá mais confiança ao pipeline de modelagem
- Torna as regras de negócio mais explícitas

### Notes
- The tolerance parameter is important for numerical comparison.
- This notebook is a validation step and should be executed before creating derived features.
<br> --- --- --- <br>
- O parâmetro de tolerância é importante para comparações numéricas.
- Este notebook é uma etapa de validação e deve ser executado antes da criação de features derivadas.

---

In [9]:
#%load_ext autoreload
#%autoreload 2

import os
import glob
from pathlib import Path
import sys
import pyarrow


PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.config.config import DATA_PROCESSED, DATA_FEATURES
from src.loaders.csv_loader import CSVLoader
from src.repository.parquet_repository import ParquetRepository

from src.dataProcessing.data_quality import DataQuality
from src.dataProcessing.data_features import DataFeatures
from src.dataProcessing.data_quality_features import DataQualityFeatures


csvLoader = CSVLoader()

In [10]:
repo = ParquetRepository(DATA_PROCESSED)

In [11]:
df = repo.load("beverage_sales_processed.parquet")

[OK] Arquivo carregado: D:\PROJETOS\git_repo\BEVERAGE-SALES\data\processed\beverage_sales_processed.parquet


In [12]:
dataQuality = DataQuality(df)

In [13]:
report = dataQuality.run_full_analysis(tolerance=0.1)
print(report)

                     rule_name issue_found  issue_count severity
0             Missing Quantity          NO            0   Medium
1            Negative Quantity          NO            0     High
2             Missing Discount          NO            0      Low
3            Negative Discount          NO            0     High
4           Discount Above One          NO            0     High
5        Invalid Customer Type          NO            0   Medium
6  B2C Discount Rule Violation          NO            0     High
7           Invalid Order Date          NO            0     High
8         Exact Duplicate Rows          NO            0   Medium
9      Total Price Consistency          NO            0     High


In [14]:
sample = dataQuality.get_total_price_inconsistencies_sample(tolerance=0.1, n=20)
print(sample.to_string(index=False))

Empty DataFrame
Columns: [Product, Region, Order_Date, Unit_Price_num, Quantity_num, Discount_num, Total_Price_num, Expected_Total, Diff]
Index: []


In [15]:
print(dataQuality.generate_executive_summary())

Data Quality Executive Summary
------------------------------
Rules evaluated: 10
Rules with issues found: 0
Total issue count: 0
High-severity issues: No high-severity issues were detected



In [16]:
#print (dataQuality.dtypes) 
print (df.dtypes) 

Order_ID          object
Customer_ID       object
Customer_Type     object
Product           object
Category          object
Unit_Price       float64
Quantity           int64
Discount         float64
Total_Price      float64
Region            object
Order_Date        object
dtype: object
